# Prepare MERFISH data for STATE SE

Run `state emb preprocess` (from the [STATE README](../../STATE/state/README.md)) to build a
gene-embedding profile. This creates `preprocessed/state_data/` with everything
the STATE SE trainer needs.

## What `state emb preprocess` does (step by step)

Starting from raw `preprocessed.h5ad` files, the command:

1. **Scans** each h5ad listed in the train & val CSVs, auto-detecting the gene-name
   field in `.var` (checks `_index`, `gene_name`, `gene_symbols`, etc.).
2. **Collects** the union of all gene names across train + val + test splits.
3. **Builds one-hot embeddings** (no ESM) — one 483-dim vector per gene
   (since MERFISH has 483 genes). Saved as `all_embeddings_{profile}.pt`.
4. **Creates per-dataset gene mappings** — for each h5ad, a tensor mapping its
   gene indices to positions in the global embedding matrix.
   Saved as `ds_emb_mapping_{profile}.torch`.
5. **Creates valid-gene masks** — boolean tensors indicating which genes in each
   dataset are present in the embedding vocabulary.
   Saved as `valid_genes_masks_{profile}.torch`.
6. **Writes updated CSV manifests** — enriched versions of the input CSVs with
   `num_cells`, `num_genes`, and `groupid_for_de` columns.
7. **Updates the YAML config** — inserts the new profile into `state_config.yaml`
   under the `embeddings.{profile}` and `dataset.{profile}` keys.

## Generated files and which are used for training

| File | Description | Used by `state emb fit`? |
|------|-------------|--------------------------|
| `all_embeddings_{profile}.pt` | One-hot gene embeddings (483 x 483 identity matrix as dict) | **Yes** — loaded as the `pe_embedding` layer in the transformer |
| `ds_emb_mapping_{profile}.torch` | Per-dataset index mapping: h5ad gene position -> embedding row | **Yes** — used by the dataloader to map each cell's genes to embedding indices |
| `valid_genes_masks_{profile}.torch` | Per-dataset boolean mask of valid genes | **Yes** — the dataloader masks out unmapped genes |
| `state_config.yaml` | Full Hydra config with profile paths, model arch, optimizer settings | **Yes** — passed as `--conf` to `state emb fit`; Hydra overrides customize it further |
| `train.csv` / `val.csv` | Input CSV manifests (species, path, names) | Only by preprocess; fit reads the enriched versions below |
| `train_{profile}.csv` / `val_{profile}.csv` | Enriched manifests with `num_cells`, `num_genes` | **Yes** — referenced in the config to locate train/val h5ad files |

In [5]:
import sys, os
sys.path.insert(0, os.path.expanduser(
    "~/noise_scaling/modeling/Scaling-up-measurement-noise-scaling-laws/scaling_laws/src"
))

from scaling_laws.prepare.data import Experiments

In [6]:
DATA_DIR = os.path.expanduser("~/noise_scaling/data")

experiments = Experiments(
    path_to_data_dir=DATA_DIR,
    datasets=["merfish"],
    qualities=[1.0],
    sizes=[60_000],
    algos=["State"],
    signal_columns=["cur_idx"],
    device=0,
)

experiments.prepare_state_data()


=== Preparing State data for merfish / 60000 / 1.0 ===
  State train manifest: /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data/train.csv
  State val manifest: /home/igor/noise_scaling/data/merfish/validation/1.0/preprocessed/state_data/val.csv
  State test manifest: /home/igor/noise_scaling/data/merfish/test/1.0/preprocessed/state_data/test.csv
  Running: /home/igor/miniconda3/envs/state/bin/python -m state emb preprocess --profile-name scaling_merfish_60000_1_0 --train-csv /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data/train.csv --val-csv /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data/val.csv --output-dir /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data --config-file /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data/state_config.yaml


2026-04-13 17:33:29,203 INFO: Loading training and validation CSV files...
2026-04-13 17:33:29,205 INFO: Processing 1 training datasets and 2 validation datasets...
2026-04-13 17:33:29,206 INFO: Scanning datasets serially...
Scanning datasets: 100%|██████████| 3/3 [00:00<00:00, 19.90it/s]
2026-04-13 17:33:29,359 INFO: Found 483 unique genes across datasets
2026-04-13 17:33:29,359 INFO: Creating one-hot embeddings...
2026-04-13 17:33:29,379 INFO: Saved embeddings to /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data/all_embeddings_scaling_merfish_60000_1_0.pt
2026-04-13 17:33:29,380 INFO: Saved dataset mapping to /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data/ds_emb_mapping_scaling_merfish_60000_1_0.torch
2026-04-13 17:33:29,380 INFO: Saved valid gene masks to /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data/valid_genes_masks_scaling_merfish_60000_1_0.torch
2026-04-13 17:33:29,479 INFO: Preprocessing completed. Run: uv

  Profile saved to /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data


In [7]:
# Verify: list the generated state_data/ contents
from pathlib import Path

state_data = Path(DATA_DIR) / "merfish" / "60000" / "1.0" / "preprocessed" / "state_data"
print(f"Profile dir: {state_data}")
for f in sorted(state_data.iterdir()):
    size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name:50s}  {size_mb:.2f} MB" if f.is_file() else f"  {f.name}/")

# Also check val / test state_data dirs were created
for split in ["validation", "test"]:
    sd = Path(DATA_DIR) / "merfish" / split / "1.0" / "preprocessed" / "state_data"
    print(f"\n{split} state_data: {sd}  (exists={sd.exists()})")
    if sd.exists():
        for f in sorted(sd.iterdir()):
            print(f"  {f.name}")

Profile dir: /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data
  all_embeddings_scaling_merfish_60000_1_0.pt         1.08 MB
  ds_emb_mapping_scaling_merfish_60000_1_0.torch      0.01 MB
  state_config.yaml                                   0.01 MB
  train.csv                                           0.00 MB
  train_scaling_merfish_60000_1_0.csv                 0.00 MB
  val.csv                                             0.00 MB
  val_scaling_merfish_60000_1_0.csv                   0.00 MB
  valid_genes_masks_scaling_merfish_60000_1_0.torch   0.00 MB

validation state_data: /home/igor/noise_scaling/data/merfish/validation/1.0/preprocessed/state_data  (exists=True)
  val.csv

test state_data: /home/igor/noise_scaling/data/merfish/test/1.0/preprocessed/state_data  (exists=True)
  test.csv


## Inspect generated files

In [8]:
import torch

state_data = Path(DATA_DIR) / "merfish" / "60000" / "1.0" / "preprocessed" / "state_data"
profile = "scaling_merfish_60000_1_0"

# 1. Gene embeddings — should be one-hot (no ESM)
embs = torch.load(state_data / f"all_embeddings_{profile}.pt", weights_only=False)
print(f"Gene embeddings: {len(embs)} genes")
first_gene = next(iter(embs))
print(f"  Example gene: '{first_gene}', vector shape: {embs[first_gene].shape}")
print(f"  Is one-hot: {embs[first_gene].sum().item() == 1.0}")

# 2. Per-dataset gene mapping
mapping = torch.load(state_data / f"ds_emb_mapping_{profile}.torch", weights_only=False)
print(f"\nDataset mappings: {list(mapping.keys())}")
for name, m in mapping.items():
    print(f"  {name}: shape {m.shape}, dtype {m.dtype}")

# 3. Valid gene masks
masks = torch.load(state_data / f"valid_genes_masks_{profile}.torch", weights_only=False)
print(f"\nValid gene masks: {list(masks.keys())}")
for name, m in masks.items():
    print(f"  {name}: {m.sum().item()}/{m.shape[0]} genes valid")

# 4. Config — show which profile is registered
import yaml
with open(state_data / "state_config.yaml") as f:
    cfg = yaml.safe_load(f)
print(f"\nConfig embeddings.{profile}:")
for k, v in cfg["embeddings"][profile].items():
    print(f"  {k}: {v}")
print(f"\nConfig dataset.{profile}:")
for k, v in cfg["dataset"][profile].items():
    print(f"  {k}: {v}")

Gene embeddings: 483 genes
  Example gene: 'ABCC9', vector shape: torch.Size([483])
  Is one-hot: True

Dataset mappings: ['scaling_merfish_60000_1_0_train', 'scaling_merfish_60000_1_0_val', 'scaling_merfish_60000_1_0_test']
  scaling_merfish_60000_1_0_train: shape torch.Size([483]), dtype torch.int64
  scaling_merfish_60000_1_0_val: shape torch.Size([483]), dtype torch.int64
  scaling_merfish_60000_1_0_test: shape torch.Size([483]), dtype torch.int64

Valid gene masks: ['scaling_merfish_60000_1_0_train', 'scaling_merfish_60000_1_0_val', 'scaling_merfish_60000_1_0_test']
  scaling_merfish_60000_1_0_train: 483/483 genes valid
  scaling_merfish_60000_1_0_val: 483/483 genes valid
  scaling_merfish_60000_1_0_test: 483/483 genes valid

Config embeddings.scaling_merfish_60000_1_0:
  all_embeddings: /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/state_data/all_embeddings_scaling_merfish_60000_1_0.pt
  ds_emb_mapping: /home/igor/noise_scaling/data/merfish/60000/1.0/preprocessed/s